In [1]:
from google.colab import files
uploaded=files.upload()

Saving reconciliation_report_day2.csv to reconciliation_report_day2.csv


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("reconciliation_report_day2.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Rows: 2000
Columns: 17


,transaction_id,order_id,customer_id,amount,payment_date,payment_status,gateway_fee,tax_on_fee,refund_amount,expected_settlement,settlement_id,settled_amount,settlement_date,difference,status,priority_score,priority
0,TXN000001,ORD000001,CUST00655,2872.14,2026-08-24,SUCCESS,32.58,5.86,0.00,2833.70,SET000001,2833.70,2026-08-26,0.0,MATCHED,0.0,LOW
1,TXN000002,ORD000002,CUST00282,6197.81,2026-08-05,SUCCESS,124.43,22.40,2526.04,3524.94,SET000002,3524.94,2026-08-08,0.0,MATCHED,0.0,LOW
2,TXN000003,ORD000003,CUST00105,16949.82,2026-08-18,SUCCESS,277.18,49.89,0.00,16622.75,SET000003,16622.75,2026-08-19,0.0,MATCHED,0.0,LOW
3,TXN000004,ORD000004,CUST00090,14803.26,2026-08-02,SUCCESS,225.37,40.57,0.00,14537.32,SET000004,14537.32,2026-08-03,0.0,MATCHED,0.0,LOW
4,TXN000005,ORD000005,CUST00031,2433.01,2026-08-08,SUCCESS,36.04,6.49,0.00,2390.48,SET000005,2390.48,2026-08-09,0.0,MATCHED,0.0,LOW


In [3]:
print(df.columns.tolist())

['transaction_id', 'order_id', 'customer_id', 'amount', 'payment_date', 'payment_status', 'gateway_fee', 'tax_on_fee', 'refund_amount', 'expected_settlement', 'settlement_id', 'settled_amount', 'settlement_date', 'difference', 'status', 'priority_score', 'priority']


In [4]:
# ----------------------------------------
# CREATE FINANCIAL FEATURES
# ----------------------------------------

df["fee_percentage"] = np.where(
    df["amount"] > 0,
    (df["gateway_fee"] / df["amount"]) * 100,
    0
)

df["refund_percentage"] = np.where(
    df["amount"] > 0,
    (df["refund_amount"] / df["amount"]) * 100,
    0
)

df["difference_percentage"] = np.where(
    df["expected_settlement"] > 0,
    (abs(df["difference"]) /
     df["expected_settlement"]) * 100,
    0
)

df["settlement_available"] = (
    df["settled_amount"].notna().astype(int)
)

print("Feature engineering completed.")

display(
    df[
        [
            "transaction_id",
            "amount",
            "gateway_fee",
            "refund_amount",
            "difference",
            "fee_percentage",
            "refund_percentage",
            "difference_percentage"
        ]
    ].head()
)

Feature engineering completed.


,transaction_id,amount,gateway_fee,refund_amount,difference,fee_percentage,refund_percentage,difference_percentage
0,TXN000001,2872.14,32.58,0.00,0.0,1.134346,0.000000,0.0
1,TXN000002,6197.81,124.43,2526.04,0.0,2.007645,40.756977,0.0
2,TXN000003,16949.82,277.18,0.00,0.0,1.635298,0.000000,0.0
3,TXN000004,14803.26,225.37,0.00,0.0,1.522435,0.000000,0.0
4,TXN000005,2433.01,36.04,0.00,0.0,1.481293,0.000000,0.0


In [5]:
feature_columns = [
    "amount",
    "gateway_fee",
    "tax_on_fee",
    "refund_amount",
    "expected_settlement",
    "difference",
    "fee_percentage",
    "refund_percentage",
    "difference_percentage",
    "settlement_available"
]

X = df[feature_columns].copy()

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(0)

print("Feature matrix:", X.shape)

Feature matrix: (2000, 10)


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Feature scaling completed.")

Feature scaling completed.


In [7]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

model.fit(X_scaled)

print("Isolation Forest trained successfully.")

Isolation Forest trained successfully.


In [8]:
df["ml_prediction"] = model.predict(X_scaled)

df["is_anomaly"] = (
    df["ml_prediction"] == -1
).astype(int)

print(
    "ML anomalies detected:",
    df["is_anomaly"].sum()
)

ML anomalies detected: 100


In [9]:
raw_scores = model.decision_function(X_scaled)

df["anomaly_score"] = -raw_scores

In [10]:
min_score = df["anomaly_score"].min()
max_score = df["anomaly_score"].max()

df["anomaly_score"] = (
    (df["anomaly_score"] - min_score)
    /
    (max_score - min_score)
) * 100

df["anomaly_score"] = (
    df["anomaly_score"].round(2)
)

print("Anomaly scores generated.")

Anomaly scores generated.


In [11]:
def calculate_ml_risk(score):

    if score >= 80:
        return "CRITICAL"

    elif score >= 60:
        return "HIGH"

    elif score >= 35:
        return "MEDIUM"

    return "LOW"


df["ml_risk"] = (
    df["anomaly_score"]
    .apply(calculate_ml_risk)
)

In [12]:
top_anomalies = (
    df
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    .head(20)
)

display(
    top_anomalies[
        [
            "transaction_id",
            "amount",
            "expected_settlement",
            "settled_amount",
            "difference",
            "status",
            "anomaly_score",
            "ml_risk"
        ]
    ]
)

,transaction_id,amount,expected_settlement,settled_amount,difference,status,anomaly_score,ml_risk
1942,TXN001943,24018.61,15797.38,NaN,15797.38,MISSING_SETTLEMENT,100.00,CRITICAL
1740,TXN001741,23274.06,22624.64,NaN,22624.64,MISSING_SETTLEMENT,99.81,CRITICAL
1594,TXN001595,23710.47,23100.34,NaN,23100.34,MISSING_SETTLEMENT,97.68,CRITICAL
1282,TXN001283,22056.09,21677.18,NaN,21677.18,MISSING_SETTLEMENT,96.97,CRITICAL
783,TXN000784,22067.90,21434.24,NaN,21434.24,MISSING_SETTLEMENT,96.68,CRITICAL
359,TXN000360,21497.62,21060.74,NaN,21060.74,MISSING_SETTLEMENT,96.60,CRITICAL
1895,TXN001896,21605.08,21249.27,NaN,21249.27,MISSING_SETTLEMENT,95.94,CRITICAL
1460,TXN001461,22629.01,22319.86,NaN,22319.86,MISSING_SETTLEMENT,94.97,CRITICAL
1693,TXN001694,20550.45,20216.36,NaN,20216.36,MISSING_SETTLEMENT,94.58,CRITICAL
866,TXN000867,20163.83,19687.74,NaN,19687.74,MISSING_SETTLEMENT,94.33,CRITICAL


In [13]:
priority_normalized = (
    df["priority_score"]
    /
    max(df["priority_score"].max(), 1)
) * 100

df["combined_risk_score"] = (
    0.6 * df["anomaly_score"]
    +
    0.4 * priority_normalized
)

df["combined_risk_score"] = (
    df["combined_risk_score"]
    .clip(0, 100)
    .round(2)
)

In [14]:
def calculate_final_risk(score):

    if score >= 80:
        return "CRITICAL"

    elif score >= 60:
        return "HIGH"

    elif score >= 35:
        return "MEDIUM"

    return "LOW"


df["risk_level"] = (
    df["combined_risk_score"]
    .apply(calculate_final_risk)
)

In [15]:
investigation_queue = df[
    df["risk_level"].isin(
        ["HIGH", "CRITICAL"]
    )
].copy()

investigation_queue = (
    investigation_queue
    .sort_values(
        "combined_risk_score",
        ascending=False
    )
)

print(
    "High/Critical cases:",
    len(investigation_queue)
)

display(
    investigation_queue[
        [
            "transaction_id",
            "amount",
            "difference",
            "status",
            "anomaly_score",
            "combined_risk_score",
            "risk_level"
        ]
    ].head(20)
)

High/Critical cases: 33


,transaction_id,amount,difference,status,anomaly_score,combined_risk_score,risk_level
1740,TXN001741,23274.06,22624.64,MISSING_SETTLEMENT,99.81,99.08,CRITICAL
1594,TXN001595,23710.47,23100.34,MISSING_SETTLEMENT,97.68,98.61,CRITICAL
1282,TXN001283,22056.09,21677.18,MISSING_SETTLEMENT,96.97,95.77,CRITICAL
1460,TXN001461,22629.01,22319.86,MISSING_SETTLEMENT,94.97,95.66,CRITICAL
783,TXN000784,22067.90,21434.24,MISSING_SETTLEMENT,96.68,95.18,CRITICAL
359,TXN000360,21497.62,21060.74,MISSING_SETTLEMENT,96.60,94.50,CRITICAL
1895,TXN001896,21605.08,21249.27,MISSING_SETTLEMENT,95.94,94.43,CRITICAL
1693,TXN001694,20550.45,20216.36,MISSING_SETTLEMENT,94.58,91.86,CRITICAL
866,TXN000867,20163.83,19687.74,MISSING_SETTLEMENT,94.33,90.81,CRITICAL
1172,TXN001173,19360.83,19034.62,MISSING_SETTLEMENT,92.63,88.69,CRITICAL


In [16]:
ml_report = df[
    [
        "transaction_id",
        "order_id",
        "customer_id",
        "amount",
        "expected_settlement",
        "settled_amount",
        "difference",
        "status",
        "priority",
        "anomaly_score",
        "ml_risk",
        "combined_risk_score",
        "risk_level"
    ]
].copy()

display(
    ml_report.head()
)

,transaction_id,order_id,customer_id,amount,expected_settlement,settled_amount,difference,status,priority,anomaly_score,ml_risk,combined_risk_score,risk_level
0,TXN000001,ORD000001,CUST00655,2872.14,2833.70,2833.70,0.0,MATCHED,LOW,15.63,LOW,9.38,LOW
1,TXN000002,ORD000002,CUST00282,6197.81,3524.94,3524.94,0.0,MATCHED,LOW,35.18,MEDIUM,21.11,LOW
2,TXN000003,ORD000003,CUST00105,16949.82,16622.75,16622.75,0.0,MATCHED,LOW,4.23,LOW,2.54,LOW
3,TXN000004,ORD000004,CUST00090,14803.26,14537.32,14537.32,0.0,MATCHED,LOW,3.27,LOW,1.96,LOW
4,TXN000005,ORD000005,CUST00031,2433.01,2390.48,2390.48,0.0,MATCHED,LOW,12.07,LOW,7.24,LOW


In [17]:
ml_report.to_csv(
    "ml_anomaly_results.csv",
    index=False
)

investigation_queue.to_csv(
    "high_risk_cases.csv",
    index=False
)

print("ML reports created successfully.")

ML reports created successfully.


In [18]:
ml_summary = pd.DataFrame({
    "metric": [
        "Total Transactions",
        "ML Anomalies",
        "Anomaly Percentage",
        "High Risk Cases",
        "Critical Risk Cases"
    ],

    "value": [
        len(df),

        int(
            df["is_anomaly"].sum()
        ),

        round(
            df["is_anomaly"].mean() * 100,
            2
        ),

        int(
            (df["risk_level"] == "HIGH").sum()
        ),

        int(
            (df["risk_level"] == "CRITICAL").sum()
        )
    ]
})

display(ml_summary)

,metric,value
0,Total Transactions,2000.0
1,ML Anomalies,100.0
2,Anomaly Percentage,5.0
3,High Risk Cases,19.0
4,Critical Risk Cases,14.0


In [19]:
ml_summary.to_csv(
    "ml_summary.csv",
    index=False
)